In [ ]:

# Phase 6 — SQL Analysis
'''**Tool:** SQLite inside Python — no Workbench needed!

We load our cleaned data into a SQL database and write real business queries.
This proves you can extract insights using SQL — a core Data Analyst skill.'''

### Queries we write:
'''| # | Business Question |
|---|---|
| Q1 | What is total revenue, orders and avg order value? |
| Q2 | Which are the top 10 revenue-generating categories? |
| Q3 | Which states have the most orders and revenue? |
| Q4 | What is the monthly revenue trend? |
| Q5 | How does delivery performance vary by state? |
| Q6 | What is the on-time vs late delivery rate? |
| Q7 | Which customer segment contributes most revenue? |
| Q8 | Who are our top 10 highest spending customers? |
| Q9 | Window function — rank categories by revenue per state |
| Q10 | CTE — find customers who ordered more than once |'''

In [ ]:
# CELL 1 — Load data into SQLite database
# sqlite3 is built into Python — no installation needed!

import sqlite3
import pandas as pd


# Load cleaned data
df  = pd.read_csv('master_cleaned.csv')
rfm = pd.read_csv('rfm_segments.csv')

# Create in-memory database and load tables
conn = sqlite3.connect('olist.db')
df.to_sql('orders', conn, if_exists='replace', index=False)
rfm.to_sql('customers', conn, if_exists='replace', index=False)

print('Database created successfully!')
print(f'Table: orders    -> {len(df):,} rows')
print(f'Table: customers -> {len(rfm):,} rows')

# Helper function — run query and show result as table
def run_query(query, title=''):
    result = pd.read_sql_query(query, conn)
    if title:
        print(f'\n{title}')
        print('-' * 50)
    print(result.to_string(index=False))
    return result

In [ ]:
# Q1 — Overall Business Performance
# What is our total revenue, number of orders, and average order value?

run_query(
    """
    SELECT
        COUNT(DISTINCT order_id)              AS total_orders,
        ROUND(SUM(total_payment_value), 2)    AS total_revenue,
        ROUND(AVG(total_payment_value), 2)    AS avg_order_value,
        ROUND(AVG(delivery_days), 1)          AS avg_delivery_days,
        ROUND(AVG(review_score), 2)           AS avg_review_score
    FROM orders
    """,
    'Q1: Overall Business KPIs'
)

In [ ]:
# Q2 — Top 10 Revenue-Generating Categories
# Which product types make us the most money?

run_query("""
    SELECT
        category,
        COUNT(order_id)                       AS total_orders,
        ROUND(SUM(total_payment_value), 2)    AS total_revenue,
        ROUND(AVG(total_payment_value), 2)    AS avg_order_value
    FROM orders
    WHERE category != 'Unknown'
    GROUP BY category
    ORDER BY total_revenue DESC
    LIMIT 10
""", 'Q2: Top 10 Categories by Revenue')

In [ ]:
# Q3 — Top States by Orders and Revenue
# Where are our best customers located?

run_query("""
    SELECT
        customer_state                        AS state,
        COUNT(order_id)                       AS total_orders,
        ROUND(SUM(total_payment_value), 2)    AS total_revenue,
        ROUND(AVG(delivery_days), 1)          AS avg_delivery_days
    FROM orders
    GROUP BY customer_state
    ORDER BY total_orders DESC
    LIMIT 10
""", 'Q3: Top 10 States by Orders')

In [ ]:
# Q4 — Monthly Revenue Trend
# Is the business growing month by month?

run_query("""
    SELECT
        order_month_yr                        AS month,
        COUNT(order_id)                       AS total_orders,
        ROUND(SUM(total_payment_value), 2)    AS monthly_revenue
    FROM orders
    GROUP BY order_month_yr
    ORDER BY order_month_yr
""", 'Q4: Monthly Revenue Trend')

In [ ]:
# Q5 — Delivery Performance by State
# Which states have the worst delivery times?

run_query("""
    SELECT
        customer_state                        AS state,
        ROUND(AVG(delivery_days), 1)          AS avg_delivery_days,
        ROUND(AVG(review_score), 2)           AS avg_review_score,
        ROUND(SUM(is_late) * 100.0
              / COUNT(order_id), 1)           AS late_delivery_pct
    FROM orders
    GROUP BY customer_state
    ORDER BY avg_delivery_days DESC
    LIMIT 10
""", 'Q5: States With Worst Delivery Performance')

In [ ]:
# Q6 — On-Time vs Late Delivery Impact
# Proves that late delivery hurts review scores

run_query("""
    SELECT
        CASE WHEN is_late = 0 THEN 'On Time'
             ELSE 'Late' END                  AS delivery_status,
        COUNT(order_id)                       AS total_orders,
        ROUND(AVG(review_score), 2)           AS avg_review_score,
        ROUND(AVG(total_payment_value), 2)    AS avg_order_value
    FROM orders
    GROUP BY is_late
    ORDER BY is_late
""", 'Q6: On-Time vs Late — Impact on Reviews')

In [ ]:
# Q7 — Revenue by Customer Segment (JOIN orders + customers)
# Which segment drives the most revenue?

run_query("""
    SELECT
        c.Segment,
        COUNT(DISTINCT c.customer_unique_id)  AS total_customers,
        ROUND(SUM(c.monetary), 2)             AS total_revenue,
        ROUND(AVG(c.monetary), 2)             AS avg_spend_per_customer,
        ROUND(AVG(c.recency), 1)              AS avg_days_since_purchase
    FROM customers c
    GROUP BY c.Segment
    ORDER BY total_revenue DESC
""", 'Q7: Revenue by RFM Customer Segment')

In [ ]:
# Q8 — Top 10 Highest Spending Customers
# Who are our most valuable individual customers?

run_query("""
    SELECT
        c.customer_unique_id,
        c.Segment,
        c.frequency                           AS total_orders,
        ROUND(c.monetary, 2)                  AS total_spent,
        c.recency                             AS days_since_last_order
    FROM customers c
    ORDER BY c.monetary DESC
    LIMIT 10
""", 'Q8: Top 10 Highest Spending Customers')

In [ ]:
# Q9 — WINDOW FUNCTION: Rank categories by revenue within each state
# Advanced SQL — shows you know window functions (asked in interviews!)

run_query("""
    SELECT *
    FROM (
        SELECT
            customer_state                    AS state,
            category,
            ROUND(SUM(total_payment_value),2) AS revenue,
            RANK() OVER (
                PARTITION BY customer_state
                ORDER BY SUM(total_payment_value) DESC
            )                                 AS rank_in_state
        FROM orders
        WHERE category != 'Unknown'
        GROUP BY customer_state, category
    )
    WHERE rank_in_state = 1
    ORDER BY revenue DESC
    LIMIT 10
""", 'Q9: Top Revenue Category in Each State (Window Function)')

In [ ]:
# Q10 — CTE: Find repeat customers vs one-time buyers
# Advanced SQL — CTE (Common Table Expression) asked in interviews!

run_query("""
    WITH customer_orders AS (
        SELECT
            customer_unique_id,
            COUNT(order_id)                   AS order_count,
            ROUND(SUM(total_payment_value),2) AS total_spent
        FROM orders
        GROUP BY customer_unique_id
    )
    SELECT
        CASE WHEN order_count = 1 THEN 'One-Time Buyer'
             ELSE 'Repeat Buyer' END          AS customer_type,
        COUNT(*)                              AS customer_count,
        ROUND(AVG(total_spent), 2)            AS avg_spend,
        ROUND(SUM(total_spent), 2)            AS total_revenue
    FROM customer_orders
    GROUP BY customer_type
    ORDER BY total_revenue DESC
""", 'Q10: Repeat vs One-Time Buyers (CTE)')

In [ ]:
# CLOSE CONNECTION AND SAVE SUMMARY

conn.close()

print('=' * 55)
print('SQL PHASE COMPLETE!')
print('=' * 55)
print('10 queries written covering:')
print('  Basic    : SELECT, WHERE, GROUP BY, ORDER BY, LIMIT')
print('  Moderate : Aggregations, CASE WHEN, JOINs')
print('  Advanced : Window Functions (RANK, PARTITION BY), CTEs')
print()
print('Interview tip: Explain Q9 and Q10 in interviews.')
print('These show advanced SQL knowledge that most freshers lack.')